# 🤖 HumanEvalComm V2 Model Benchmarking System

A comprehensive framework for evaluating language models on coding tasks with advanced communication and code quality metrics.

**Version:** V2.0  
**Date:** 2025-09-15  
**Authors:** HumanEvalComm Team

---

## 📋 Table of Contents

1. [Overview](#Overview)
2. [V2 Metrics Framework](#V2-Metrics-Framework)
3. [Setup & Installation](#Setup-&-Installation)
4. [Data Preparation](#Data-Preparation)
5. [Running Evaluations](#Running-Evaluations)
6. [Results Analysis](#Results-Analysis)
7. [Visualization Dashboard](#Visualization-Dashboard)
8. [Advanced Usage](#Advanced-Usage)
9. [Troubleshooting](#Troubleshooting)

---

## 🎯 Overview

The HumanEvalComm V2 Benchmarking System provides a comprehensive evaluation framework for language models on coding tasks. Unlike traditional code generation benchmarks that only measure correctness, V2 evaluates:

- **Communication Skills**: How well models ask clarifying questions
- **Code Quality**: Readability, maintainability, and security
- **Efficiency**: Runtime and memory performance
- **Reliability**: Judge consensus and calibration
- **Overall Performance**: Weighted composite scores

### Key Features

✅ **15+ Evaluation Metrics** covering all aspects of code generation  
✅ **Multi-Model Support** for comparative benchmarking  
✅ **Automated Pipeline** with static/dynamic analysis and LLM judging  
✅ **Interactive Visualizations** and leaderboards  
✅ **Communication Analysis** from evaluation logs  
✅ **Configurable Weights** for custom scoring formulas  

### Example Use Case

Compare GPT-4, Claude, and open-source models on coding tasks with ambiguous requirements to see which models:
- Ask the most relevant clarifying questions
- Produce the most readable and secure code
- Perform best under resource constraints

## 📊 V2 Metrics Framework

The V2 framework evaluates models across 6 dimensions with 15+ specific metrics:

### 1. Core Communication Metrics
- **Communication Rate (%)**: How often the model asks clarifying questions
- **Good Question Rate (%)**: Percentage of questions judged useful by humans
- **Clarification Efficiency**: Average questions asked before final code (lower = better)

### 2. Code Correctness
- **Pass@1 (%)**: First-attempt solutions that pass all tests
- **Test Pass Rate (%)**: Average fraction of test cases passed
- **Fuzz Test Robustness (%)**: Additional correctness via property-based testing

### 3. Code Trustworthiness
- **Readability Score (0-100)**: Pylint + cyclomatic complexity
- **Maintainability Index (0-100)**: Comment density and complexity balance
- **Security Score (0-100)**: Bandit vulnerability scanning

### 4. Efficiency
- **Efficiency (Normalized 0-1)**: Combined runtime/memory performance
- **Runtime (sec)**: Actual execution time
- **Peak Memory (MB)**: Memory usage during execution

### 5. Reliability Indicators
- **Judge Consensus Confidence (%)**: Agreement among LLM judges
- **Calibration Gap (%)**: Difference between predictions and ground truth

### 6. Composite Score
- **HumanEvalComm V2 Score (0-100)**: Weighted average across all metrics

### Default Weights
```python
evaluation_weights = {
    'correctness': 0.40,      # Code working correctly
    'communication': 0.20,    # Asking good questions
    'readability': 0.15,      # Code readability
    'security': 0.10,         # Security vulnerabilities
    'efficiency': 0.10,       # Performance
    'maintainability': 0.05   # Long-term maintainability
}
```

## 🛠️ Setup & Installation

### Prerequisites

- Python 3.11+
- Docker (for sandboxed execution)
- Git

### Installation Steps

In [ ]:
# Clone the repository
!git clone https://github.com/your-org/human-eval-comm.git
!cd human-eval-comm

# Create virtual environment
!python -m venv venv
!source venv/bin/activate  # On Windows: venv\Scripts\activate

# Install dependencies
!pip install -r requirements.txt

# Verify installation
import sys
print(f"Python version: {sys.version}")
print("✅ Environment ready!")

### Configuration

The system uses `config.yaml` for all settings. Key configurations:

In [ ]:
import yaml

# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("🔧 Current Configuration:")
print(f"Judge Models: {len(config['judge_models'])}")
print(f"Weights: {config['evaluation_weights']}")
print(f"Sandbox: {config['sandbox']['use_docker']}")

## 📝 Data Preparation

### Input Format

Models are evaluated using JSONL files with the following format:

In [ ]:
import json

# Example model data structure
sample_data = [
    {
        "model_name": "GPT-4",
        "code": """def fibonacci(n):\n    if n < 0:\n        raise ValueError('n must be non-negative')\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)""",
        "test_code": """# Test cases here""",
        "problem_description": "Implement fibonacci function with error handling",
        "problem_id": "fibonacci_gpt4"
    },
    {
        "model_name": "Okanagan", 
        "code": """# Different implementation""",
        "test_code": """# Test cases""",
        "problem_description": "Same problem",
        "problem_id": "fibonacci_okanagan"
    }
]

# Save to JSONL file
with open('benchmark_data.jsonl', 'w') as f:
    for item in sample_data:
        f.write(json.dumps(item) + '\n')

print("📄 Sample data saved to benchmark_data.jsonl")
print(f"Models to evaluate: {[d['model_name'] for d in sample_data]}")

### Loading Existing Data

If you have existing evaluation results:

In [ ]:
import pandas as pd

# Load existing results
try:
    results_df = pd.read_json('evaluation_results.jsonl', lines=True)
    print(f"📊 Loaded {len(results_df)} evaluation results")
    print(f"Models: {results_df['model_name'].unique()}")
    display(results_df.head())
except FileNotFoundError:
    print("❌ No existing results found. Run evaluation first.")

## 🚀 Running Evaluations

### Single Model Evaluation

In [ ]:
from evaluators.automated_static_dynamic import AutomatedStaticDynamic
from evaluators.sandbox_runner import SandboxRunner
from evaluators.multi_llm_judge import MultiLLMJudge
from evaluators.enhanced_aggregator import EnhancedAggregator

# Initialize components
analyzer = AutomatedStaticDynamic()
sandbox = SandboxRunner(use_docker=False)
llm_judge = MultiLLMJudge('config.yaml')
aggregator = EnhancedAggregator()

print("🔧 Components initialized")

# Example evaluation
test_code = """
def fibonacci(n):
    '''Calculate nth Fibonacci number'''
    if n < 0:
        raise ValueError("n must be non-negative")
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# Test cases
assert fibonacci(0) == 0
assert fibonacci(1) == 1
assert fibonacci(5) == 5
print("All tests passed!")
"""

test_cases = """
def test_fibonacci():
    assert fibonacci(0) == 0
    assert fibonacci(1) == 1
    assert fibonacci(10) == 55
"""

print("🧪 Running evaluation pipeline...")

# 1. Static Analysis
static_results, dynamic_results = analyzer.analyze_code(test_code, test_cases, "fibonacci_test")
print(f"📏 Static Analysis - Pylint: {static_results.pylint_score:.1f}")

# 2. Sandbox Execution
sandbox_results = sandbox.run_code(test_code, test_cases)
print(f"🏃 Execution: {'✅ Success' if sandbox_results.success else '❌ Failed'}")
print(f"⏱️  Runtime: {sandbox_results.execution_time:.2f}s")

# 3. LLM Evaluation
import asyncio
llm_scores = asyncio.run(llm_judge.evaluate_code(test_code, "Implement fibonacci with error handling"))
print(f"🧠 LLM Score: {llm_scores.consensus_score:.1f}/10")

# 4. Aggregation
evaluation_data = {
    'problem_id': 'fibonacci_test',
    'static_results': static_results.__dict__,
    'dynamic_results': dynamic_results.__dict__,
    'sandbox_results': sandbox_results.__dict__,
    'llm_scores': llm_scores.__dict__,
}

final_score = aggregator.aggregate_results(evaluation_data)
print(f"🎯 V2 Composite Score: {final_score.composite_score:.1f}")

### Multi-Model Benchmarking

In [ ]:
# Run the complete benchmarking pipeline
import subprocess
import sys

def run_benchmarking(data_file):
    """Run the benchmarking script"""
    cmd = [sys.executable, 'scripts/benchmark_models.py', '--data', data_file]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ Benchmarking completed successfully!")
        print(result.stdout)
    else:
        print("❌ Benchmarking failed:")
        print(result.stderr)
    
    return result.returncode == 0

# Run benchmarking on sample data
success = run_benchmarking('benchmark_test_data.jsonl')

if success:
    print("\n📊 Results generated:")
    print("- evaluation_results.jsonl")
    print("- results/leaderboard.csv")
    print("- results/dashboard.html")

## 📈 Results Analysis

### Loading and Exploring Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# Load evaluation results
results_df = pd.read_json('evaluation_results.jsonl', lines=True)
print(f"📊 Loaded {len(results_df)} evaluation results")
print(f"Models evaluated: {results_df['model_name'].unique()}")
print(f"Problems: {results_df['problem_id'].unique()}")

# Display summary
display(results_df.head())

### Key Metrics Summary

In [ ]:
# Calculate summary statistics by model
summary_stats = results_df.groupby('model_name').agg({
    'v2_composite_score': ['mean', 'std', 'min', 'max'],
    'pass_at_1': 'mean',
    'readability_100': 'mean',
    'security_100': 'mean',
    'efficiency_normalized': 'mean',
    'judge_consensus_confidence': 'mean'
}).round(2)

print("📈 Model Performance Summary:")
display(summary_stats)

### Leaderboard Generation

In [ ]:
# Load the generated leaderboard
leaderboard_df = pd.read_csv('results/leaderboard.csv')
print("🏆 V2 Leaderboard:")
display(leaderboard_df)

# Sort by V2 Score for ranking
leaderboard_df = leaderboard_df.sort_values('V2 Score', ascending=False)
leaderboard_df['Rank'] = range(1, len(leaderboard_df) + 1)
leaderboard_df = leaderboard_df[['Rank'] + [col for col in leaderboard_df.columns if col != 'Rank']]

print("\n🥇 Ranked Leaderboard:")
display(leaderboard_df)

## 📊 Visualization Dashboard

### Radar Chart: Multi-Dimensional Performance

In [ ]:
import numpy as np
from math import pi

def create_radar_chart(df, metrics, title):
    """Create radar chart for model comparison"""
    
    # Number of variables
    categories = list(metrics.keys())
    N = len(categories)
    
    # Angle for each axis
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]  # Close the loop
    
    # Initialize figure
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
    
    # Plot each model
    for idx, row in df.iterrows():
        values = [row[metric] for metric in metrics.values()]
        values += values[:1]  # Close the loop
        
        ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'])
        ax.fill(angles, values, alpha=0.25)
    
    # Labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 100)
    ax.set_title(title, size=16, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0))
    ax.grid(True)
    
    plt.tight_layout()
    return fig

# Define metrics for radar chart
radar_metrics = {
    'Readability': 'Readability Score',
    'Security': 'Security Score', 
    'Efficiency': 'Efficiency',
    'Reliability': 'Judge Consensus Confidence',
    'V2 Score': 'V2 Score'
}

# Create radar chart
radar_fig = create_radar_chart(leaderboard_df, radar_metrics, 'Model Performance Comparison')
plt.show()

### Bar Chart: Metric Comparison

In [ ]:
# Create bar chart comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('V2 Metrics Comparison by Model', fontsize=16, fontweight='bold')

metrics_to_plot = [
    ('V2 Score', 'V2 Score'),
    ('Readability Score', 'Readability Score'),
    ('Security Score', 'Security Score'),
    ('Efficiency', 'Efficiency'),
    ('Judge Consensus Confidence', 'Judge Consensus Confidence'),
    ('Test Pass Rate', 'Test Pass Rate')
]

for idx, (title, column) in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    
    bars = ax.bar(leaderboard_df['Model'], leaderboard_df[column])
    ax.set_title(title)
    ax.set_ylabel('Score')
    
    # Rotate x labels if needed
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

### Correlation Analysis

In [ ]:
# Calculate correlations between metrics
correlation_metrics = [
    'V2 Score', 'Readability Score', 'Security Score', 'Efficiency',
    'Judge Consensus Confidence', 'Test Pass Rate', 'Pass@1'
]

corr_matrix = leaderboard_df[correlation_metrics].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Metric Correlations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔍 Key Correlations:")
print(f"V2 Score vs Readability: {corr_matrix.loc['V2 Score', 'Readability Score']:.3f}")
print(f"V2 Score vs Security: {corr_matrix.loc['V2 Score', 'Security Score']:.3f}")
print(f"V2 Score vs Efficiency: {corr_matrix.loc['V2 Score', 'Efficiency']:.3f}")

## 🔧 Advanced Usage

### Custom Evaluation Weights

In [ ]:
# Modify evaluation weights for different priorities
custom_weights = {
    'correctness': 0.50,      # Prioritize correctness
    'communication': 0.10,    # Less emphasis on communication
    'readability': 0.20,      # More emphasis on readability
    'security': 0.10,
    'efficiency': 0.05,
    'maintainability': 0.05
}

print("⚖️ Custom Weights Configuration:")
for category, weight in custom_weights.items():
    print(f"{category.capitalize()}: {weight*100:.0f}%")

# To apply custom weights, modify config.yaml
print("\n📝 To use custom weights, update config.yaml:")
print("evaluation_weights:")
for k, v in custom_weights.items():
    print(f"  {k}: {v}")

### Communication Metrics Analysis

If you have communication logs from the evaluation process:

In [ ]:
# Analyze communication patterns from logs
from scripts.export_leaderboard import parse_comm_metrics

# Parse communication metrics from log files
comm_metrics = parse_comm_metrics('log')

print("💬 Communication Metrics by Model:")
for model, metrics in comm_metrics.items():
    print(f"\n{model}:")
    print(f"  Communication Rate: {metrics.comm_rate():.1f}%")
    print(f"  Good Question Rate: {metrics.good_q_rate():.1f}%")
    print(f"  Clarification Efficiency: {metrics.clarification_efficiency():.2f}")
    print(f"  Total Questions: {metrics.question_entries}")
    print(f"  Problems: {metrics.problems_count}")

### Export Results for External Analysis

In [ ]:
# Export results in different formats

# 1. CSV for spreadsheet analysis
leaderboard_df.to_csv('v2_leaderboard_detailed.csv', index=False)
print("📄 Exported detailed CSV: v2_leaderboard_detailed.csv")

# 2. JSON for programmatic access
results_df.to_json('evaluation_results_formatted.json', orient='records', indent=2)
print("📄 Exported JSON: evaluation_results_formatted.json")

# 3. Summary statistics
summary = {
    'total_models': len(leaderboard_df),
    'total_evaluations': len(results_df),
    'top_model': leaderboard_df.iloc[0]['Model'],
    'top_score': leaderboard_df.iloc[0]['V2 Score'],
    'average_score': leaderboard_df['V2 Score'].mean(),
    'score_std': leaderboard_df['V2 Score'].std()
}

import json
with open('benchmark_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("📄 Exported summary: benchmark_summary.json")
print("\n📊 Benchmark Summary:")
for key, value in summary.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

## 🐛 Troubleshooting

### Common Issues and Solutions

#### 1. Docker Issues
```bash
# Check Docker status
docker --version
docker ps

# For subprocess mode instead of Docker
# Edit config.yaml: sandbox.use_docker: false
```

#### 2. LLM API Errors
```bash
# Check API keys in environment
echo $OPENAI_API_KEY
echo $GEMINI_API_KEY

# Test API connectivity
python -c "import openai; print('OpenAI OK')"
```

#### 3. Import Errors
```bash
# Ensure virtual environment is activated
source venv/bin/activate

# Reinstall dependencies
pip install -r requirements.txt
```

#### 4. Memory Issues
- Reduce batch size in evaluations
- Use subprocess instead of Docker
- Close other applications

#### 5. Slow Performance
- LLM API calls are the bottleneck
- Consider using cached results
- Reduce number of judge models in config

### Getting Help

1. Check the logs in `v2_evaluation.log`
2. Verify your `config.yaml` settings
3. Test individual components separately
4. Review the BENCHMARKING_README.md

---

## 📚 Additional Resources

- [HumanEvalComm Paper](https://arxiv.org/abs/...) - Original research
- [V2 Framework Details](docs/v2_framework.md) - Technical specification
- [API Reference](docs/api.md) - Complete API documentation
- [Examples](examples/) - Sample evaluations and data

---

**🎉 Happy Benchmarking!** 

*For questions or contributions, please open an issue on GitHub.*